# Querying ZTF cutouts — validation notebook

Validates the query half of `ztfcomet` on a single test target (**24P/Schaumasse**).
For production runs use `notebooks/main.py`; this notebook exists to check each
stage in isolation and to inspect what comes back.

Covered here:

1. Resolve paths and the target from `ztfcomet.config` — no hardcoded paths, no pasted orbit IDs.
2. Coarse ephemeris → IRSA image search → footprint containment test.
3. Build cutout URLs and download them **with validation**.
4. Confirm every downloaded file really is FITS.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Make the package importable from a checkout, wherever the kernel started.
_root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
              if (p / "ztfcomet" / "__init__.py").is_file()), None)
if _root and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import numpy as np
import pandas as pd
import ztfcomet as zc

zc.setup_logging()          # the pipeline reports dropped epochs and bad downloads via logging
print(zc.directory.describe())

## 1. Target and configuration

Everything target-specific lives in `ztfcomet/config.py`. The predecessor
pasted orbit record numbers into notebook cells, which is how
`ztfquery_2P.ipynb` ended up annotating its figures with 24P's ephemeris.

In [ ]:
target = zc.get_target("24P")

print(f"name        : {target.name}")
print(f"designation : {target.query_designation}")
print(f"fragments   : {'allowed' if target.allow_fragment else 'excluded'}")
print(f"window      : {target.start_date} .. {target.end_date}")
print(f"note        : {target.note}")
print()
print(target.query)

In [ ]:
# Narrow the window so this notebook runs in a couple of minutes.
# `with_dates` returns a copy; the entry in config.TARGETS is untouched.
test = target.with_dates("2025-09-01", "2025-10-31")

DATADIR = zc.data_dir(test.name)
FIGDIR  = zc.fig_dir(test.name)
print(DATADIR)

### Fragments and stale record numbers

Two traps sit between a designation and the right ephemeris, and both silently
return the **wrong object**:

**Fragments share the parent's designation.** Asking Horizons for `240P` gives
an ambiguity listing containing `240P` (two orbit solutions) *and* `240P-B`, a
fragment. Taking the last record — what the predecessor did — selects the
fragment, which is ~2.9 mag fainter.

**Record numbers are not stable.** Horizons renumbers its small-body records.
The numbers the pre-merge notebooks hardcoded for 240P (`90001203`, `90001204`)
today resolve to **233P/La Sagra** and **234P/LINEAR**.

`ztfcomet.horizons` resolves by designation, drops fragments, picks the orbit
solution nearest the observation, and verifies `targetname` on the way back.

In [ ]:
from ztfcomet import horizons as hz

t240 = zc.get_target("240P")
for jd, label in [(2458300.5, "2018 apparition"), (2460900.5, "2025 apparition")]:
    rid = t240.resolve_orbit_record(jd)
    print(f"  {label}: 240P -> Horizons record {rid}")

print(f"\n  fragment requested explicitly: "
      f"{zc.get_target('240P-B').resolve_orbit_record(2460900.5)}")

In [ ]:
# The guard that catches a wrong object however it got through.
for returned, requested in [("240P/NEAT", "240P"), ("240P-B/NEAT", "240P"),
                            ("234P/LINEAR", "240P")]:
    ok, why = hz.verify_targetname(returned, requested)
    print(f"  {returned:14s} for {requested:6s} -> {'OK' if ok else 'REJECTED'}  {why[:60]}")

## 2. Ephemeris and image search

`search_frames` runs the whole search and returns a `QueryReport` alongside the
data. The report matters: the predecessor wrapped its loop in a bare
`except Exception: print(...)`, so a network failure and genuine absence of ZTF
coverage were indistinguishable and neither was recorded.

In [ ]:
eph, frames, report = zc.search_frames(test)

print(report)
if report.failures:
    print("\nFAILURES:")
    for line in report.failures[:10]:
        print(" ", line)

In [ ]:
# The coarse ephemeris, after the rh / Vmag cuts.
eph[["datetime_str", "datetime_jd", "RA", "DEC", "r", "Tmag"]].head()

In [ ]:
# Frames whose footprint contains the target.
cols = ["obsjd", "filtercode", "field", "ccdid", "qid", "seeing", "maglimit",
        "RA", "DEC", "r", "delta", "alpha"]
frames[[c for c in cols if c in frames.columns]].head(10)

In [ ]:
print(f"frames found : {len(frames)}")
print(f"by filter    : {frames['filtercode'].value_counts().to_dict()}")
print(f"date range   : {frames['obsdate'].min()}  ..  {frames['obsdate'].max()}")

# The ephemeris block is joined on JD, not by row position. Verify it stuck:
assert frames["RA"].notna().all(), "some frames have no ephemeris"
print("\nephemeris joined to every frame — OK")

### Check the JD join explicitly

The predecessor used `pd.concat(axis=1)`, which lines the two tables up by row
position. That was correct only because Horizons happens to return epochs sorted
and the caller happened to sort first. Here the join is on JD, so it can be
checked directly.

In [ ]:
# Every frame's ephemeris must correspond to its own observation time.
dt_seconds = (frames["obsjd"] - frames["datetime_jd"]).abs() * 86400
print(f"max |obsjd - ephemeris jd| = {dt_seconds.max():.3f} s")
assert dt_seconds.max() < 1.0, "ephemeris is attached to the wrong frame!"

# And no duplicated column labels (the predecessor carried 'airmass' twice).
dupes = frames.columns[frames.columns.duplicated()].tolist()
print(f"duplicate column labels    : {dupes or 'none'}")
assert not dupes

## 3. Build and save cutout URLs

In [ ]:
urls = zc.build_urls(frames,
                     is_cutout=test.query.is_cutout,
                     cutout_size=test.query.cutout_size)

print(urls["fits_url"].iloc[0])
print()
print(f"{len(urls)} URLs")

eph.to_csv(DATADIR / "eph.csv", index=False)
urls.to_csv(DATADIR / "ztf.csv", index=False)
zc.save_urls(urls, DATADIR / "fits_urls.txt")

## 4. Download, with validation

IRSA answers some cutout requests with HTTP 200 and an HTML error body. The
predecessor wrote those to disk as `*_sciimg.fits` — 245-byte files reading
`<title>404 Not Found</title>` — and its resume check (`if exists: continue`)
meant no later run could repair them.

`download_urls` checks the payload really is FITS, writes through a `.part`
file, and records per-URL status in `download_manifest.csv`. Pass `repair=True`
(the default) to re-fetch anything corrupt from an earlier run.

In [ ]:
report = zc.download_urls(urls, DATADIR, repair=True)
print(report)

for url, why in report.failures[:5]:
    print(f"  FAILED {url.split('/')[-1][:60]} :: {why}")

In [ ]:
# Independent check: every .fits on disk starts with the FITS magic.
paths = sorted(DATADIR.glob("*.fits"))
bad = [p.name for p in paths if p.open("rb").read(6) != b"SIMPLE"]

print(f"{len(paths)} files on disk, {len(bad)} not valid FITS")
assert not bad, bad
print("all downloads are genuine FITS — OK")

## 5. Quick look

Full figure work lives in `figure.ipynb`; this is just a sanity check that the
cutouts contain what we asked for.

In [ ]:
import matplotlib.pyplot as plt

table = zc.build_frame_table(DATADIR)
table = zc.attach_ephemerides(table, test)
table.insert(1, "target", test.name)

ax = zc.plot_cutout(DATADIR / table.iloc[len(table) // 2]["file"],
                    row=table.iloc[len(table) // 2],
                    target=test, show_apertures=False)
plt.show()

In [ ]:
table[["file", "isot", "filter", "exptime", "fwhm_pix", "maglim",
       "zpmag", "clrcoeff", "r", "delta", "alpha"]].head(10)

## Next

- `afrho.ipynb` — photometry and Afρ on these frames.
- `figure.ipynb` — publication figures.
- `notebooks/main.py 24P` — the whole chain in one command.